## Cell 0A — Google Drive Mount
Run this first, every Colab session.
Safe to re-run — uses `force_remount=False`.

In [ ]:
import sys, subprocess

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    try:
        drive.mount('/content/drive', force_remount=False)
        print('Drive mounted at /content/drive')
    except Exception as _e:
        if 'already contain files' in str(_e) or 'symlink' in str(_e):
            print('Stale mount detected — clearing and remounting...')
            subprocess.run(['umount', '/content/drive'], capture_output=True)
            subprocess.run(['rm', '-rf', '/content/drive'], capture_output=True)
            drive.mount('/content/drive', force_remount=False)
            print('Drive remounted successfully.')
        else:
            raise
else:
    print('Not in Colab — skipping Drive mount.')


## Cell 0B — Install Dependencies

In [ ]:
import subprocess, sys

packages = ['pyarrow', 'opencv-python-headless']
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet'] + packages,
    capture_output=True, text=True
)
if result.returncode != 0:
    print(result.stdout)
    print(result.stderr)
    raise RuntimeError('Dependency installation failed. See output above.')

print('Dependencies ready:', ', '.join(packages))


## Cell 0C — Path Configuration

In [ ]:
from pathlib import Path

# ── EDIT THIS LINE ────────────────────────────────────────────────────────────
DRIVE_PROJECT_PATH = 'SIGNAL_NN_2026/HIDRO_2026'   # folder inside MyDrive
# ─────────────────────────────────────────────────────────────────────────────

if IN_COLAB:
    DRIVE_ROOT = Path('/content/drive/MyDrive') / DRIVE_PROJECT_PATH
else:
    DRIVE_ROOT = Path.home() / 'BubbleFlow'

print('DRIVE_ROOT   :', DRIVE_ROOT)
print('Drive exists :', DRIVE_ROOT.exists())


---
# NB-00-pre · Reference Frame Extraction
### Bubble Flow Analysis System — Stage 0 (pre-calibration)

**Run once per physical setup** — before NB-00.

**Purpose:** Extract one representative frame from the actual experiment video
and save it as `reference_frame.jpg` in the setup folder. NB-00 uses this
file as `REFERENCE_FRAME_PATH` for checkerboard detection and ROI definition.

**Why this matters:**
- The reference frame must have the same resolution, orientation, and geometry
  as the experiment videos.
- Using a photo taken from a different device or distance produces ROI
  coordinates that are wrong for the video — causing SNR failures in NB-01.
- This notebook guarantees the reference frame is always correct.

**Gate:** Frame is portrait (H > W), dimensions match video, file saved to Drive.

---
### Run order
```
NB-00-pre  →  NB-00  →  NB-01  →  NB-02  →  NB-03  →  NB-04  →  NB-05  →  NB-06
```

---
## Cell 1 — Configuration
**Edit only this cell.**

In [ ]:
# ── SOURCE VIDEO ──────────────────────────────────────────────────────────────
# Any video from this campaign works — the frame geometry is the same for all.
# Prefer a frame where bubbles are active AND the checkerboard is visible.
# If the checkerboard was filmed separately, use that video instead.
VIDEO_PATH : str = str(DRIVE_ROOT / 'campaigns/setup_A/run_001_Q10/VID_20260413_124627.mp4')

# ── FRAME SELECTION ───────────────────────────────────────────────────────────
# Frame index to extract. Choose a frame where:
#   - The checkerboard is clearly visible (for NB-00 calibration)
#   - Bubbles are actively rising (to confirm the column is lit)
#   - There is no strong motion blur
FRAME_IDX : int = 10   # early frame — checkerboard typically shown at start

# ── OUTPUT ────────────────────────────────────────────────────────────────────
# This file becomes REFERENCE_FRAME_PATH in NB-00 Cell 1.
# Saved to the SETUP folder — shared across all runs in the campaign.
SETUP_DIR  : str = str(DRIVE_ROOT / 'campaigns/setup_A')
OUTPUT_NAME: str = 'reference_frame.png'   # PNG: lossless, no JPEG artifacts on edges

# ── SAVE FORMAT ───────────────────────────────────────────────────────────────
# 'png'  — lossless. Recommended — preserves checkerboard edges exactly.
# 'jpg'  — lossy. Use only if Drive storage is a concern.
SAVE_FORMAT    : str = 'png'
JPEG_QUALITY   : int = 97   # used only if SAVE_FORMAT = 'jpg'
PNG_COMPRESSION: int = 3    # 0 = no compression, 9 = max. 3 is a safe default.

# ── CAMPAIGN LABEL ────────────────────────────────────────────────────────────
# Used in provenance record. Must match RUN_LABEL in NB-02 through NB-06.
RUN_LABEL : str = 'run_001_Q10'
SETUP_ID  : str = 'setup_A'

# ── DISPLAY ───────────────────────────────────────────────────────────────────
# Scale factor for display only (does NOT affect the saved file)
DISPLAY_SCALE: float = 0.15   # 0.15 × 3840 = 576 px tall — fits Colab output

# ── GATE FLAGS ────────────────────────────────────────────────────────────────
# Set False to disable a specific gate check (e.g. for landscape test videos)
REQUIRE_VIDEO_EXISTS          : bool = True
REQUIRE_PORTRAIT_ORIENTATION  : bool = True
REQUIRE_METADATA_JSON         : bool = True

# ── PREPARE FOLDERS ───────────────────────────────────────────────────────────
SETUP_PATH  = Path(SETUP_DIR)
SETUP_PATH.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = SETUP_PATH / OUTPUT_NAME

print('Video         :', VIDEO_PATH)
print('Frame index   :', FRAME_IDX)
print('Output file   :', OUTPUT_PATH)
print('Save format   :', SAVE_FORMAT)
print('Setup dir     :', SETUP_PATH, '  exists =', SETUP_PATH.exists())
print('Run label     :', RUN_LABEL)
print('Gates         : video_exists=' + str(REQUIRE_VIDEO_EXISTS)
      + '  portrait=' + str(REQUIRE_PORTRAIT_ORIENTATION)
      + '  metadata=' + str(REQUIRE_METADATA_JSON))


---
## Cell 2 — Extract Frame and Verify Orientation

In [ ]:
import cv2
import numpy as np

video_path = Path(VIDEO_PATH)
if REQUIRE_VIDEO_EXISTS and not video_path.exists():
    raise FileNotFoundError(
        'Video not found: ' + str(video_path) + '\n'
        'Set VIDEO_PATH in Cell 1 to the full path of any experiment video.'
    )

cap = cv2.VideoCapture(str(video_path))
if not cap.isOpened():
    raise IOError('cv2.VideoCapture could not open: ' + str(video_path))

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
v_fps        = cap.get(cv2.CAP_PROP_FPS)
v_w          = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
v_h          = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print('Video metadata')
print('  File         :', video_path.name)
print('  Resolution   : W=' + str(v_w) + '  H=' + str(v_h))
print('  Orientation  :', 'portrait (correct)' if v_h > v_w else 'landscape — check camera rotation')
print('  FPS          :', round(v_fps, 4))
print('  Total frames :', total_frames)
print('  Duration     :', round(total_frames / v_fps, 2) if v_fps else 'unknown', 's')

if FRAME_IDX >= total_frames:
    raise ValueError(
        'FRAME_IDX=' + str(FRAME_IDX) + ' exceeds total frames (' + str(total_frames) + ').\n'
        'Set FRAME_IDX to a value between 0 and ' + str(total_frames - 1) + '.'
    )

cap.set(cv2.CAP_PROP_POS_FRAMES, FRAME_IDX)
ret, frame = cap.read()
cap.release()

if not ret or frame is None:
    raise IOError('Could not read frame ' + str(FRAME_IDX) + ' from ' + str(video_path))

f_h, f_w = frame.shape[:2]
print()
print('Extracted frame')
print('  Frame index  :', FRAME_IDX)
print('  Shape (H, W) : (' + str(f_h) + ', ' + str(f_w) + ')')
print('  dtype        :', frame.dtype)
print('  min / max    :', int(frame.min()), '/', int(frame.max()))
print('  Orientation  :', 'portrait' if f_h > f_w else 'landscape')

if REQUIRE_PORTRAIT_ORIENTATION and f_h <= f_w:
    raise ValueError(
        'Gate FAIL: expected portrait (H > W) but got W=' + str(f_w) + ', H=' + str(f_h) + '.\n'
        'Set REQUIRE_PORTRAIT_ORIENTATION = False if landscape is intentional.'
    )
else:
    print()
    print('Orientation gate: PASS (portrait)')


---
## Cell 3 — Display Frame with Pixel Grid
Read the pixel coordinates of the column body from this image.
You will need them for the ROI definition in NB-00 Cell 6 (headless override).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Display the frame with a pixel coordinate grid overlaid
# Grid spacing in original pixel coordinates
GRID_STEP_X = 200   # px
GRID_STEP_Y = 400   # px

fig, ax = plt.subplots(figsize=(6, 10))
ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

# Overlay grid lines
for x in range(0, f_w, GRID_STEP_X):
    ax.axvline(x, color='cyan', lw=0.5, alpha=0.6)
    ax.text(x + 4, 30, str(x), color='cyan', fontsize=7, va='top')
for y in range(0, f_h, GRID_STEP_Y):
    ax.axhline(y, color='cyan', lw=0.5, alpha=0.6)
    ax.text(8, y + 8, str(y), color='cyan', fontsize=7, va='top')

ax.set_title(
    'Reference frame  (frame ' + str(FRAME_IDX) + ')\n'
    'Read x0, y0, x1, y1 from the cyan grid for the ROI.\n'
    'Column body: find the bright vertical region and note its x and y bounds.',
    fontsize=9
)
ax.axis('off')
plt.tight_layout()
plt.show()

print('Grid step: every ' + str(GRID_STEP_X) + ' px horizontally, ' + str(GRID_STEP_Y) + ' px vertically.')
print('Frame size: W=' + str(f_w) + '  H=' + str(f_h))
print()
print('Read the ROI coordinates from the grid and note them:')
print('  x0 = left  edge of column body')
print('  y0 = top   edge of active region (below free surface)')
print('  x1 = right edge of column body')
print('  y1 = bottom edge (above sparger plate)')
print()
print('Enter them in NB-00 Cell 24 (headless ROI override):')
print('  roi_coords      = [x0, y0, x1, y1]')
print('  reference_lines = [y0 + (y1-y0)*0.25, y0 + (y1-y0)*0.50, y0 + (y1-y0)*0.75]')

---
## Cell 4 — Save Reference Frame to Drive

In [ ]:
# Save the frame in the format set in Cell 1.
# PNG is the default — lossless, preserves checkerboard edge detail exactly.
if SAVE_FORMAT in {'jpg', 'jpeg'}:
    encode_params = [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY]
else:
    encode_params = [cv2.IMWRITE_PNG_COMPRESSION, PNG_COMPRESSION]

success = cv2.imwrite(str(OUTPUT_PATH), frame, encode_params)

if not success:
    raise IOError(
        'cv2.imwrite failed — check OUTPUT_PATH and Drive permissions.\n'
        'Path attempted: ' + str(OUTPUT_PATH)
    )

size_kb = OUTPUT_PATH.stat().st_size // 1024
print('Reference frame saved:')
print('  Path     :', OUTPUT_PATH)
print('  Size     :', size_kb, 'KB')
print('  Shape    : W=' + str(f_w) + '  H=' + str(f_h))
print('  Format   :', SAVE_FORMAT.upper())
print()
print('Next step: open NB-00 and set in Cell 1:')
print("  REFERENCE_FRAME_PATH = '" + str(OUTPUT_PATH) + "'")


---
## Cell 5 — Gate and Summary

In [ ]:
from IPython.display import Markdown, display
from datetime import datetime, timezone
import json

# ── Gate evaluation ───────────────────────────────────────────────────────────
gate_file_saved = OUTPUT_PATH.exists()
gate_portrait   = (f_h > f_w) if REQUIRE_PORTRAIT_ORIENTATION else True

# ── Provenance record ─────────────────────────────────────────────────────────
# Stored alongside the reference frame so downstream notebooks can audit
# exactly which video and frame produced this calibration reference.
provenance_path = SETUP_PATH / (OUTPUT_PATH.stem + '_provenance.json')

provenance = {
    'notebook':          'NB-00-pre',
    'setup_id':          SETUP_ID,
    'run_label':         RUN_LABEL,
    'source_video':      str(video_path),
    'frame_index':       FRAME_IDX,
    'frame_shape_hw':    [f_h, f_w],
    'orientation':       'portrait' if f_h > f_w else 'landscape',
    'video_fps':         round(v_fps, 6),
    'video_frames':      total_frames,
    'saved_to':          str(OUTPUT_PATH),
    'save_format':       SAVE_FORMAT,
    'created_at_utc':    datetime.now(timezone.utc).isoformat(),
    'gates': {
        'file_saved':           gate_file_saved,
        'portrait_orientation': gate_portrait,
        'metadata_json':        True,
    },
}
provenance_path.write_text(json.dumps(provenance, indent=2), encoding='utf-8')

gate_metadata = provenance_path.exists() if REQUIRE_METADATA_JSON else True
gate_pass     = gate_file_saved and gate_portrait and gate_metadata

# ── Print gate result ─────────────────────────────────────────────────────────
print('=' * 58)
print('NB-00-pre  ·  GATE')
print('=' * 58)
print('  File saved          :', gate_file_saved)
print('  Portrait orientation:', gate_portrait)
print('  Metadata JSON       :', gate_metadata, '->', provenance_path.name)
print('  Gate                :', '✅  PASS' if gate_pass else '❌  FAIL')
print('=' * 58)
print()

display(Markdown(
    '## NB-00-pre  ·  Reference Frame Extraction Complete\n\n'
    '| Field | Value |\n|---|---|\n'
    '| **Reference frame** | `' + str(OUTPUT_PATH) + '` |\n'
    '| **Provenance JSON** | `' + str(provenance_path) + '` |\n'
    '| **Frame index** | ' + str(FRAME_IDX) + ' |\n'
    '| **Resolution** | W=' + str(f_w) + '  H=' + str(f_h) + ' |\n'
    '| **Orientation** | ' + ('portrait ✅' if f_h > f_w else 'landscape ⚠️') + ' |\n'
    '| **Format** | ' + SAVE_FORMAT.upper() + ' |\n'
    '| **Gate** | ' + ('✅  PASS' if gate_pass else '❌  FAIL') + ' |\n\n'
    '**Next step:** Open `NB-00` and set in Cell 1:\n\n'
    '```python\n'
    "REFERENCE_FRAME_PATH = '" + str(OUTPUT_PATH) + "'\n"
    '```\n\n'
    'NB-00 will use this portrait frame for checkerboard detection and ROI definition. '
    'The resulting `config.json` will have `roi_coords` valid for all experiment videos.'
))
